In [ ]:
# ============================================================
# COACH — Session Start  (do not remove this cell)
# ============================================================
import sys, os
sys.path.insert(0, os.path.expanduser('~/Desktop/applied-ai-research'))
from coach.notebook_widgets import render_session_start
_SESSION = render_session_start(
    module_id="06-video-recommendation",
    notebook_name="03_ranking_models.ipynb"
)

# Ranking Models Deep Dive

## The Big Idea (For a 12-Year-Old)

In the previous notebook, we **grabbed a few thousand videos** from the billions available. That was the candidate generation stage -- fast but rough.

Now comes the **careful selection** stage. Imagine you have 1,000 books on a table and need to pick the BEST 50. This time, you can actually **read the back cover, check the reviews, see how similar books you liked before** -- because you only have 1,000 books, not 10 billion!

This notebook covers:
1. **Content-Based Ranking Model** -- using rich features to score each candidate
2. **Feature Interactions** -- why combining features matters more than individual features
3. **Diversity Enforcement** -- making sure recommendations are not all the same genre
4. **Re-Ranking Service** -- business rules, freshness, regional restrictions
5. **Complete Scoring Pipeline** -- putting it all together

---

## Staff-Level Technical Summary

- **Ranking model**: DNN-based pointwise scorer using concatenated user + video feature vectors
- **Feature interactions**: Cross features, deep and cross networks, feature interaction learning
- **Multi-objective**: Relevance + diversity as dual optimization targets
- **Diversity**: Pairwise cosine similarity, maximal marginal relevance (MMR)
- **Re-ranking**: Rules-based post-processing for business constraints

## 1. Content-Based Ranking Model

### Why Content-Based for Ranking?

The candidate generation stage uses **collaborative filtering** (fast, interaction-based). But for ranking, we switch to **content-based** because:

1. We only have ~1,000 candidates (not billions) -- we can afford a heavy model
2. We can use ALL features: video title, tags, duration, user demographics, context, history
3. Content-based models produce more accurate relevance scores

Think of it as: CF says "roughly who would like what" and content-based says "precisely HOW MUCH would this specific user like this specific video."

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# === Content-Based Ranking Model ===

class VideoRankingModel(nn.Module):
    """
    Content-Based Ranking Model for Video Recommendation.
    
    12-year-old version:
    This model is like a very smart judge at a talent show.
    It looks at EVERYTHING about the contestant (the video)
    AND EVERYTHING about the judge (the user) to score
    how much this specific judge would like this specific act.
    
    Staff-level detail:
    - Takes concatenated user + video feature vector as input
    - Deep neural network with batch norm, dropout, and skip connections
    - Output: single relevance score (sigmoid probability)
    - Loss: Binary Cross Entropy (relevant vs. not relevant)
    
    Key difference from two-tower:
    - Two-tower: separate encoders, dot-product similarity (fast but limited)
    - Ranking model: JOINT encoding of user+video features (slow but expressive)
    - The ranking model can learn CROSS-FEATURE interactions:
      e.g., 'users aged 18-25 who use mobile prefer short gaming videos'
    """
    def __init__(self, input_dim, hidden_dims=[512, 256, 128, 64]):
        super().__init__()
        
        layers = []
        prev_dim = input_dim
        
        for i, hidden_dim in enumerate(hidden_dims):
            layers.extend([
                nn.Linear(prev_dim, hidden_dim),
                nn.BatchNorm1d(hidden_dim),
                nn.ReLU(),
                nn.Dropout(0.3 if i < 2 else 0.1)  # More dropout in early layers
            ])
            prev_dim = hidden_dim
        
        # Final prediction layer
        layers.append(nn.Linear(prev_dim, 1))
        layers.append(nn.Sigmoid())
        
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x).squeeze(-1)


# Display architecture
print("=== Content-Based Ranking Model Architecture ===")
print()
print("  INPUT: Concatenated User + Video Features")
print("  ============================================")
print("  User Features:")
print("    - User ID embedding (64d)")
print("    - Age embedding (8d)")
print("    - Gender embedding (4d)")
print("    - Language embedding (16d)")
print("    - Time of day (4d)")
print("    - Device type (4d)")
print("    - Avg search history embedding (64d)")
print("    - Avg liked videos embedding (64d)")
print("    - Avg watched videos embedding (64d)")
print("  Video Features:")
print("    - Video ID embedding (64d)")
print("    - Title BERT embedding (128d)")
print("    - Tags CBOW embedding (64d)")
print("    - Duration (1d)")
print("    - Language embedding (16d)")
print("    - Likes count (1d)")
print("    - Views count (1d)")
print("  ============================================")
print("  Total input: ~558 dimensions")
print()
print("  ARCHITECTURE:")
print("    Concat(558) -> Linear(512) -> BN -> ReLU -> Dropout(0.3)")
print("                -> Linear(256) -> BN -> ReLU -> Dropout(0.3)")
print("                -> Linear(128) -> BN -> ReLU -> Dropout(0.1)")
print("                -> Linear(64)  -> BN -> ReLU -> Dropout(0.1)")
print("                -> Linear(1)   -> Sigmoid")
print()
print("  OUTPUT: P(relevant) in [0, 1]")

In [ ]:
# === Training the Ranking Model ===

np.random.seed(42)
torch.manual_seed(42)

# Simulate feature vectors for (user, video) pairs
n_samples = 10000
input_dim = 558  # Total concatenated feature dimension

# Feature names for interpretability
feature_groups = {
    'User ID emb': 64,
    'Age emb': 8,
    'Gender emb': 4,
    'User language emb': 16,
    'Time of day': 4,
    'Device type': 4,
    'Search history emb': 64,
    'Liked videos emb': 64,
    'Watched videos emb': 64,
    'Video ID emb': 64,
    'Title BERT emb': 128,
    'Tags CBOW emb': 64,
    'Duration': 1,
    'Video language emb': 16,
    'Likes count': 1,
    'Views count': 1,
    'Like-to-view ratio': 1
}
actual_dim = sum(feature_groups.values())

# Generate synthetic features
X = np.random.randn(n_samples, actual_dim).astype(np.float32)

# Generate labels with realistic patterns
# Relevance is higher when:
# - User and video language match (simulate via dot product of language embeddings)
# - Video has high like-to-view ratio
# - User's watch history is similar to video content
offset = 0
for name, dim in feature_groups.items():
    offset += dim

user_lang_start = 64 + 8 + 4  # After User ID, Age, Gender
video_lang_start = sum(list(feature_groups.values())[:13])  # Video language position

lang_match = np.sum(X[:, user_lang_start:user_lang_start+16] * 
                    X[:, video_lang_start:video_lang_start+16], axis=1)
like_ratio = X[:, -1]  # Last feature

logits = -1.5 + 0.3 * lang_match + 0.2 * like_ratio + 0.1 * np.random.randn(n_samples)
probs = 1 / (1 + np.exp(-logits))
y = (np.random.rand(n_samples) < probs).astype(np.float32)

print(f"Dataset: {n_samples} (user, video) pairs, {actual_dim} features")
print(f"Positive (relevant): {y.sum():.0f} ({y.mean()*100:.1f}%)")
print(f"Negative: {(1-y).sum():.0f} ({(1-y.mean())*100:.1f}%)")
print(f"\nFeature groups:")
for name, dim in feature_groups.items():
    print(f"  {name}: {dim}d")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create model
model = VideoRankingModel(input_dim=actual_dim, hidden_dims=[512, 256, 128, 64])
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-5)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=5, gamma=0.5)

# Data loaders
train_dataset = TensorDataset(torch.FloatTensor(X_train), torch.FloatTensor(y_train))
test_dataset = TensorDataset(torch.FloatTensor(X_test), torch.FloatTensor(y_test))
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256)

# Training loop
print("=== Training the Ranking Model ===")
print(f"Architecture: {actual_dim} -> 512 -> 256 -> 128 -> 64 -> 1")
print(f"Loss: Binary Cross Entropy | Optimizer: Adam (lr=0.001)")
print()

train_losses = []
test_aucs = []

for epoch in range(20):
    # Train
    model.train()
    epoch_loss = 0
    n_batches = 0
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        preds = model(batch_X)
        loss = criterion(preds, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        n_batches += 1
    
    scheduler.step()
    avg_loss = epoch_loss / n_batches
    train_losses.append(avg_loss)
    
    # Evaluate
    model.eval()
    with torch.no_grad():
        test_preds = model(torch.FloatTensor(X_test)).numpy()
        test_auc = roc_auc_score(y_test, test_preds)
        test_aucs.append(test_auc)
    
    if (epoch + 1) % 4 == 0:
        print(f"  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f} | Test AUC: {test_auc:.4f}")

print(f"\nFinal Test AUC: {test_aucs[-1]:.4f}")
print(f"Final Test AP:  {average_precision_score(y_test, test_preds):.4f}")

In [ ]:
# === Visualize Training Progress ===

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(train_losses, 'b-', linewidth=2)
axes[0].set_xlabel('Epoch', fontsize=12)
axes[0].set_ylabel('BCE Loss', fontsize=12)
axes[0].set_title('Training Loss', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

axes[1].plot(test_aucs, 'r-o', linewidth=2, markersize=4)
axes[1].set_xlabel('Epoch', fontsize=12)
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title('Test AUC-ROC', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(y=0.5, color='gray', linestyle='--', label='Random baseline')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

## 2. Feature Interaction Learning

### Why Feature Interactions Matter

Individual features are useful, but **combinations of features** are even MORE powerful.

For example:
- "User is 20 years old" -- somewhat useful
- "Video is a gaming video" -- somewhat useful
- "User is 20 years old AND video is gaming AND it's Saturday night AND they're on mobile" -- VERY predictive!

The deep neural network learns these interactions automatically in its hidden layers.

In [ ]:
# === Demonstrating Feature Interactions ===

# Let's create a simple example showing why interactions matter

np.random.seed(42)
n = 2000

# Two features:
# - user_age_bucket: 0=teen, 1=adult, 2=senior
# - video_type: 0=gaming, 1=news, 2=cooking
user_age = np.random.randint(0, 3, n)
video_type = np.random.randint(0, 3, n)

# True relevance depends on INTERACTION:
# Teens like gaming, adults like news, seniors like cooking
interaction_relevance = np.array([
    [0.9, 0.1, 0.2],  # Teens: love gaming, meh on news/cooking
    [0.3, 0.8, 0.4],  # Adults: like news, some gaming, some cooking
    [0.1, 0.5, 0.9],  # Seniors: love cooking, like news, skip gaming
])

y_interaction = np.array([interaction_relevance[a, v] for a, v in zip(user_age, video_type)])
y_labels = (y_interaction + np.random.randn(n) * 0.15 > 0.5).astype(float)

# Model 1: No interaction features (just age + video type)
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder

# One-hot encode
X_no_interaction = np.column_stack([
    np.eye(3)[user_age],     # 3 age features
    np.eye(3)[video_type]    # 3 video type features
])

# Model 2: WITH interaction features (age x video type)
X_with_interaction = np.column_stack([
    X_no_interaction,
    np.eye(9)[user_age * 3 + video_type]  # 9 cross features
])

# Train and evaluate both
from sklearn.model_selection import cross_val_score

lr_no_cross = LogisticRegression(max_iter=1000)
lr_with_cross = LogisticRegression(max_iter=1000)

scores_no = cross_val_score(lr_no_cross, X_no_interaction, y_labels, cv=5, scoring='roc_auc')
scores_with = cross_val_score(lr_with_cross, X_with_interaction, y_labels, cv=5, scoring='roc_auc')

print("=== Feature Interaction Experiment ===")
print(f"\nTrue pattern: Teens like gaming, Adults like news, Seniors like cooking")
print(f"This pattern requires INTERACTION between age and video type to learn!\n")

print(f"  Without interactions (6 features):  AUC = {scores_no.mean():.4f} (+/- {scores_no.std():.4f})")
print(f"  WITH interactions (15 features):    AUC = {scores_with.mean():.4f} (+/- {scores_with.std():.4f})")
print(f"\n  Improvement: {(scores_with.mean() - scores_no.mean())*100:.1f}% absolute AUC gain")
print(f"\n  Key insight: DNNs learn these interactions AUTOMATICALLY in hidden layers.")
print(f"  That's why deep ranking models outperform simple linear models.")

In [ ]:
# === Visualize the interaction pattern ===

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Heatmap of true interaction patterns
im = axes[0].imshow(interaction_relevance, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
axes[0].set_xticks([0, 1, 2])
axes[0].set_xticklabels(['Gaming', 'News', 'Cooking'], fontsize=11)
axes[0].set_yticks([0, 1, 2])
axes[0].set_yticklabels(['Teen', 'Adult', 'Senior'], fontsize=11)
axes[0].set_xlabel('Video Type', fontsize=12)
axes[0].set_ylabel('User Age Group', fontsize=12)
axes[0].set_title('True Relevance\n(Age x Video Type Interaction)', fontsize=13, fontweight='bold')

for i in range(3):
    for j in range(3):
        axes[0].text(j, i, f'{interaction_relevance[i,j]:.1f}',
                    ha='center', va='center', fontsize=14, fontweight='bold',
                    color='white' if interaction_relevance[i,j] > 0.5 else 'black')

plt.colorbar(im, ax=axes[0])

# Bar chart comparing models
model_names = ['Without\nInteractions', 'WITH\nInteractions']
model_aucs = [scores_no.mean(), scores_with.mean()]
model_stds = [scores_no.std(), scores_with.std()]
colors = ['#ff7043', '#66bb6a']

bars = axes[1].bar(model_names, model_aucs, yerr=model_stds, capsize=5,
                   color=colors, edgecolor='black')
axes[1].set_ylabel('AUC-ROC', fontsize=12)
axes[1].set_title('Impact of Feature Interactions', fontsize=13, fontweight='bold')
axes[1].set_ylim(0.4, 1.0)
axes[1].axhline(y=0.5, color='gray', linestyle='--', alpha=0.5)

for bar, auc in zip(bars, model_aucs):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
                f'{auc:.3f}', ha='center', fontsize=12, fontweight='bold')

plt.tight_layout()
plt.show()

## 3. Diversity in Recommendations

### The Echo Chamber Problem

If you only optimize for relevance, you might recommend 20 dog videos to someone who likes dogs. That is relevant... but BORING. Users want variety!

**Diversity** measures how different the recommended videos are from each other. We compute it using **pairwise cosine similarity** between video embeddings.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# === Diversity Computation ===

def compute_diversity(video_embeddings):
    """
    Compute diversity of a set of recommended videos.
    
    12-year-old version:
    Compare every video to every other video. If they're all super
    similar (all dog videos), diversity is LOW. If they're all different
    (dogs, cooking, gaming, music), diversity is HIGH.
    
    Staff-level detail:
    - Compute pairwise cosine similarity matrix
    - Average the upper triangle (excluding diagonal)
    - Low average similarity = high diversity
    - Diversity = 1 - avg_similarity
    """
    sim_matrix = cosine_similarity(video_embeddings)
    n = len(video_embeddings)
    
    # Get upper triangle (exclude diagonal)
    upper_tri = sim_matrix[np.triu_indices(n, k=1)]
    avg_similarity = np.mean(upper_tri)
    diversity = 1 - avg_similarity
    
    return diversity, avg_similarity, sim_matrix


# Create two sets of recommendations:
# Set A: All similar videos (low diversity)
# Set B: Diverse videos (high diversity)

np.random.seed(42)
embedding_dim = 32
n_recs = 8

# Set A: All dog/animal videos (similar embeddings with small noise)
base_animal_emb = np.random.randn(embedding_dim)
set_a = np.array([base_animal_emb + np.random.randn(embedding_dim) * 0.3 for _ in range(n_recs)])
set_a_names = ['Dog Playing', 'Cat Funny', 'Puppy Training', 'Kitten Sleeping',
               'Dog Tricks', 'Pet Care', 'Animal Rescue', 'Dog Show']

# Set B: Diverse videos (different base embeddings)
set_b = np.random.randn(n_recs, embedding_dim)  # Independent random embeddings
set_b_names = ['Dog Playing', 'Python Tutorial', 'Cooking Show', 'Basketball Game',
               'Music Concert', 'Travel Vlog', 'News Analysis', 'Comedy Sketch']

# Compute diversity for both
div_a, sim_a, sim_matrix_a = compute_diversity(set_a)
div_b, sim_b, sim_matrix_b = compute_diversity(set_b)

print("=== Diversity Comparison ===")
print(f"\nSet A (all animal videos):")
print(f"  Videos: {', '.join(set_a_names[:4])}...")
print(f"  Avg Pairwise Similarity: {sim_a:.4f}")
print(f"  Diversity Score: {div_a:.4f} (LOW -- echo chamber!)")

print(f"\nSet B (diverse videos):")
print(f"  Videos: {', '.join(set_b_names[:4])}...")
print(f"  Avg Pairwise Similarity: {sim_b:.4f}")
print(f"  Diversity Score: {div_b:.4f} (HIGH -- good variety!)")

In [ ]:
# === Visualize Diversity ===

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Similarity heatmap for Set A
im1 = axes[0].imshow(sim_matrix_a, cmap='YlOrRd', vmin=-0.5, vmax=1, aspect='auto')
axes[0].set_xticks(range(n_recs))
axes[0].set_xticklabels(set_a_names, rotation=45, ha='right', fontsize=8)
axes[0].set_yticks(range(n_recs))
axes[0].set_yticklabels(set_a_names, fontsize=8)
axes[0].set_title(f'Set A: LOW Diversity ({div_a:.2f})\n(All Animal Videos = Echo Chamber)',
                  fontsize=11, fontweight='bold', color='red')
plt.colorbar(im1, ax=axes[0])

# Similarity heatmap for Set B
im2 = axes[1].imshow(sim_matrix_b, cmap='YlOrRd', vmin=-0.5, vmax=1, aspect='auto')
axes[1].set_xticks(range(n_recs))
axes[1].set_xticklabels(set_b_names, rotation=45, ha='right', fontsize=8)
axes[1].set_yticks(range(n_recs))
axes[1].set_yticklabels(set_b_names, fontsize=8)
axes[1].set_title(f'Set B: HIGH Diversity ({div_b:.2f})\n(Mixed Topics = Good Variety)',
                  fontsize=11, fontweight='bold', color='green')
plt.colorbar(im2, ax=axes[1])

plt.tight_layout()
plt.show()

print("\nWarning: Diverse but IRRELEVANT is useless!")
print("Always balance diversity WITH relevance metrics (mAP, precision@k).")

In [ ]:
# === Maximal Marginal Relevance (MMR) for Diversity-Aware Ranking ===

def mmr_rerank(relevance_scores, video_embeddings, lambda_param=0.5, top_k=5):
    """
    Maximal Marginal Relevance: balances relevance and diversity.
    
    12-year-old version:
    Pick videos one at a time. Each time you pick one, you consider:
    1. How relevant is this video? (high score = good)
    2. How DIFFERENT is it from videos you already picked? (more different = good)
    
    lambda controls the balance:
    - lambda=1.0: only care about relevance (no diversity)
    - lambda=0.0: only care about diversity (no relevance)
    - lambda=0.5: equal balance (recommended)
    
    Staff-level detail:
    MMR(i) = lambda * relevance(i) - (1-lambda) * max_similarity(i, selected)
    At each step, select the document with highest MMR score.
    """
    n = len(relevance_scores)
    selected = []
    remaining = list(range(n))
    
    # Normalize relevance scores to [0, 1]
    rel_norm = (relevance_scores - relevance_scores.min()) / \
               (relevance_scores.max() - relevance_scores.min() + 1e-8)
    
    # Similarity matrix
    sim_matrix = cosine_similarity(video_embeddings)
    
    for _ in range(min(top_k, n)):
        if not selected:
            # First pick: highest relevance
            best = max(remaining, key=lambda i: rel_norm[i])
        else:
            best = None
            best_mmr = -float('inf')
            
            for i in remaining:
                # Max similarity to any already-selected video
                max_sim = max(sim_matrix[i][j] for j in selected)
                
                # MMR score
                mmr = lambda_param * rel_norm[i] - (1 - lambda_param) * max_sim
                
                if mmr > best_mmr:
                    best_mmr = mmr
                    best = i
        
        selected.append(best)
        remaining.remove(best)
    
    return selected


# Demo: Compare pure relevance ranking vs MMR
np.random.seed(42)
n_candidates = 20

# Create video embeddings in 3 clusters (3 genres)
genre_centers = np.random.randn(3, 16) * 2
genre_labels = np.random.randint(0, 3, n_candidates)
genre_names_map = {0: 'Gaming', 1: 'Cooking', 2: 'Music'}

candidate_embeddings = np.array([
    genre_centers[g] + np.random.randn(16) * 0.3 for g in genre_labels
])

# Relevance scores (gaming videos happen to score highest)
relevance = np.array([0.9 - 0.2 * g + np.random.rand() * 0.15 for g in genre_labels])

# Pure relevance ranking
pure_top5 = np.argsort(-relevance)[:5].tolist()

# MMR ranking
mmr_top5 = mmr_rerank(relevance, candidate_embeddings, lambda_param=0.5, top_k=5)

print("=== Pure Relevance vs MMR Ranking ===")
print(f"\nPure Relevance Top 5 (no diversity consideration):")
pure_genres = [genre_names_map[genre_labels[i]] for i in pure_top5]
for rank, idx in enumerate(pure_top5, 1):
    print(f"  #{rank}: Video {idx} ({genre_names_map[genre_labels[idx]]}) - relevance: {relevance[idx]:.3f}")
print(f"  Genre distribution: {dict(zip(*np.unique(pure_genres, return_counts=True)))}")

print(f"\nMMR Top 5 (balanced relevance + diversity):")
mmr_genres = [genre_names_map[genre_labels[i]] for i in mmr_top5]
for rank, idx in enumerate(mmr_top5, 1):
    print(f"  #{rank}: Video {idx} ({genre_names_map[genre_labels[idx]]}) - relevance: {relevance[idx]:.3f}")
print(f"  Genre distribution: {dict(zip(*np.unique(mmr_genres, return_counts=True)))}")

# Compute diversity for both
div_pure, _, _ = compute_diversity(candidate_embeddings[pure_top5])
div_mmr, _, _ = compute_diversity(candidate_embeddings[mmr_top5])
print(f"\nDiversity (pure relevance): {div_pure:.4f}")
print(f"Diversity (MMR):            {div_mmr:.4f}")
print(f"\nMMR gives {(div_mmr - div_pure)/div_pure*100:.0f}% more diversity while keeping top videos!")

## 4. Re-Ranking Service

### Why Re-Ranking?

The scoring model optimizes for relevance (and maybe diversity). But there are **business rules** that ML cannot capture:

- Remove videos that violate content policies
- Regional restrictions (some videos cannot be shown in certain countries)
- Freshness boost (new content should get exposure)
- Creator diversity (do not show 10 videos from the same creator)
- Already-watched filtering

In [ ]:
# === Re-Ranking Service ===

class ReRankingService:
    """
    Post-processing service that applies business rules to ranked results.
    
    12-year-old version:
    The ML model picked the 'best' videos, but a human manager needs
    to check the final list:
    - 'Wait, this video is banned in their country!'
    - 'We showed them this video yesterday, remove it!'
    - 'All 10 videos are from the same creator -- spread them out!'
    - 'This new video needs some exposure, boost it!'
    """
    
    def __init__(self):
        self.rules_applied = []
    
    def apply_content_policy(self, videos, user_region='US'):
        """Remove flagged/inappropriate content."""
        original_count = len(videos)
        filtered = [v for v in videos if not v.get('flagged', False)]
        removed = original_count - len(filtered)
        if removed > 0:
            self.rules_applied.append(f"Content policy: removed {removed} flagged videos")
        return filtered
    
    def apply_regional_restrictions(self, videos, user_region='US'):
        """Remove videos restricted in user's region."""
        original_count = len(videos)
        filtered = [v for v in videos if user_region not in v.get('restricted_regions', [])]
        removed = original_count - len(filtered)
        if removed > 0:
            self.rules_applied.append(f"Regional: removed {removed} restricted videos")
        return filtered
    
    def apply_freshness_boost(self, videos, boost_factor=1.2, age_threshold_days=3):
        """Boost scores of recently uploaded videos."""
        boosted_count = 0
        for v in videos:
            if v.get('age_days', 999) <= age_threshold_days:
                v['score'] *= boost_factor
                v['freshness_boosted'] = True
                boosted_count += 1
        if boosted_count > 0:
            self.rules_applied.append(f"Freshness: boosted {boosted_count} new videos by {boost_factor}x")
        return videos
    
    def enforce_creator_diversity(self, videos, max_per_creator=2):
        """Limit videos from the same creator."""
        creator_counts = {}
        result = []
        demoted = 0
        deferred = []
        
        for v in videos:
            creator = v.get('creator', 'unknown')
            creator_counts[creator] = creator_counts.get(creator, 0) + 1
            if creator_counts[creator] <= max_per_creator:
                result.append(v)
            else:
                deferred.append(v)
                demoted += 1
        
        result.extend(deferred)  # Add demoted videos at the end
        if demoted > 0:
            self.rules_applied.append(f"Creator diversity: demoted {demoted} videos (max {max_per_creator}/creator)")
        return result
    
    def remove_already_watched(self, videos, watched_ids):
        """Remove videos the user has already watched."""
        original_count = len(videos)
        filtered = [v for v in videos if v['video_id'] not in watched_ids]
        removed = original_count - len(filtered)
        if removed > 0:
            self.rules_applied.append(f"Already watched: removed {removed} videos")
        return filtered
    
    def rerank(self, videos, user_region='US', watched_ids=None):
        """Apply all re-ranking rules in sequence."""
        self.rules_applied = []
        
        videos = self.apply_content_policy(videos, user_region)
        videos = self.apply_regional_restrictions(videos, user_region)
        videos = self.remove_already_watched(videos, watched_ids or set())
        videos = self.apply_freshness_boost(videos)
        videos = self.enforce_creator_diversity(videos)
        
        # Re-sort by (potentially modified) score
        videos.sort(key=lambda v: v['score'], reverse=True)
        
        return videos


# === Demo: Re-Ranking in Action ===

np.random.seed(42)
ranked_videos = [
    {'video_id': 1, 'title': 'Amazing Dog Tricks', 'score': 0.95, 'creator': 'PetLover', 'age_days': 1, 'flagged': False, 'restricted_regions': []},
    {'video_id': 2, 'title': 'Top 10 Dog Breeds', 'score': 0.92, 'creator': 'PetLover', 'age_days': 30, 'flagged': False, 'restricted_regions': []},
    {'video_id': 3, 'title': 'Puppy Training 101', 'score': 0.90, 'creator': 'PetLover', 'age_days': 15, 'flagged': False, 'restricted_regions': []},
    {'video_id': 4, 'title': 'Cat vs Cucumber', 'score': 0.88, 'creator': 'FunnyAnimals', 'age_days': 2, 'flagged': False, 'restricted_regions': []},
    {'video_id': 5, 'title': 'Python Tutorial', 'score': 0.85, 'creator': 'TechGuru', 'age_days': 5, 'flagged': False, 'restricted_regions': []},
    {'video_id': 6, 'title': 'Controversial Take', 'score': 0.83, 'creator': 'EdgeLord', 'age_days': 1, 'flagged': True, 'restricted_regions': []},
    {'video_id': 7, 'title': 'UK News Update', 'score': 0.80, 'creator': 'BBC', 'age_days': 0, 'flagged': False, 'restricted_regions': ['US', 'JP']},
    {'video_id': 8, 'title': 'Cooking Pasta', 'score': 0.78, 'creator': 'ChefMario', 'age_days': 2, 'flagged': False, 'restricted_regions': []},
    {'video_id': 9, 'title': 'Gaming Highlights', 'score': 0.75, 'creator': 'GamePro', 'age_days': 1, 'flagged': False, 'restricted_regions': []},
    {'video_id': 10, 'title': 'Travel Japan', 'score': 0.72, 'creator': 'Wanderlust', 'age_days': 10, 'flagged': False, 'restricted_regions': []},
]

print("=== BEFORE Re-Ranking ===")
for v in ranked_videos:
    flags = []
    if v['flagged']: flags.append('FLAGGED')
    if v['restricted_regions']: flags.append(f'restricted:{v["restricted_regions"]}')
    flag_str = f" [{', '.join(flags)}]" if flags else ""
    print(f"  #{v['video_id']:2d} | {v['score']:.2f} | {v['creator']:15s} | {v['title']}{flag_str}")

# Apply re-ranking
reranker = ReRankingService()
already_watched = {2}  # User already watched video 2
final_videos = reranker.rerank(ranked_videos, user_region='US', watched_ids=already_watched)

print(f"\n=== Rules Applied ===")
for rule in reranker.rules_applied:
    print(f"  - {rule}")

print(f"\n=== AFTER Re-Ranking ===")
for rank, v in enumerate(final_videos, 1):
    boost = " [FRESH!]" if v.get('freshness_boosted') else ""
    print(f"  #{rank:2d} | {v['score']:.2f} | {v['creator']:15s} | {v['title']}{boost}")

## 5. Complete Scoring Pipeline Simulation

In [ ]:
# === End-to-End Ranking Pipeline ===

class VideoRankingPipeline:
    """
    Complete ranking pipeline that takes candidate videos and produces
    final recommendations.
    
    Pipeline:
    1. Feature computation (user + video features for each pair)
    2. Model scoring (predict relevance for each candidate)
    3. Diversity optimization (MMR re-ranking)
    4. Business rules (re-ranking service)
    5. Final top-k selection
    """
    
    def __init__(self, model, reranker):
        self.model = model
        self.reranker = reranker
    
    def compute_features(self, user_profile, video_metadata):
        """
        Compute feature vector for a (user, video) pair.
        
        In production, this pulls from:
        - Feature store (precomputed, static features)
        - Real-time computation (dynamic features like time-of-day)
        """
        # Simulate feature vector (in production, these are real features)
        np.random.seed(hash((user_profile['user_id'], video_metadata['video_id'])) % 2**31)
        return np.random.randn(actual_dim).astype(np.float32)
    
    def score_candidates(self, user_profile, candidates):
        """Score all candidates using the ranking model."""
        features = np.array([
            self.compute_features(user_profile, v) for v in candidates
        ])
        
        self.model.eval()
        with torch.no_grad():
            scores = self.model(torch.FloatTensor(features)).numpy()
        
        for v, score in zip(candidates, scores):
            v['score'] = float(score)
        
        # Sort by score
        candidates.sort(key=lambda v: v['score'], reverse=True)
        return candidates
    
    def rank(self, user_profile, candidates, top_k=10):
        """Complete ranking pipeline."""
        print(f"  Step 1: Received {len(candidates)} candidates from candidate generation")
        
        # Score
        scored = self.score_candidates(user_profile, candidates)
        print(f"  Step 2: Scored all candidates (top score: {scored[0]['score']:.4f})")
        
        # Take top candidates for re-ranking
        top_candidates = scored[:top_k * 3]  # Take 3x for re-ranking headroom
        
        # Re-rank
        final = self.reranker.rerank(
            top_candidates,
            user_region=user_profile.get('region', 'US'),
            watched_ids=user_profile.get('watched_ids', set())
        )
        print(f"  Step 3: Applied re-ranking rules")
        for rule in self.reranker.rules_applied:
            print(f"    - {rule}")
        
        return final[:top_k]


# === Simulate the full pipeline ===

# Create candidate videos
np.random.seed(42)
categories = ['Gaming', 'Cooking', 'Music', 'Education', 'Sports', 'Comedy', 'News', 'Travel']
creators = ['Creator_A', 'Creator_B', 'Creator_C', 'Creator_D', 'Creator_E',
            'Creator_F', 'Creator_G', 'Creator_H']

candidates = []
for i in range(50):  # 50 candidates from candidate generation
    candidates.append({
        'video_id': i + 100,
        'title': f'{np.random.choice(categories)} Video #{i}',
        'category': np.random.choice(categories),
        'creator': np.random.choice(creators),
        'age_days': np.random.randint(0, 30),
        'flagged': np.random.random() < 0.05,  # 5% flagged
        'restricted_regions': ['JP'] if np.random.random() < 0.1 else [],
    })

# User profile
user_profile = {
    'user_id': 42,
    'name': 'Alice',
    'age': 25,
    'region': 'US',
    'watched_ids': {100, 105, 110}  # Already watched these
}

# Run pipeline
pipeline = VideoRankingPipeline(model, ReRankingService())

print(f"=== Ranking Pipeline for {user_profile['name']} ===")
final_recs = pipeline.rank(user_profile, candidates, top_k=10)

print(f"\n=== Final Recommendations (Top 10) ===")
for rank, v in enumerate(final_recs, 1):
    boost = " [FRESH]" if v.get('freshness_boosted') else ""
    print(f"  #{rank:2d} | Score: {v['score']:.4f} | {v['creator']:12s} | {v['title']}{boost}")

In [ ]:
# === Full System Architecture Diagram ===

fig, ax = plt.subplots(figsize=(14, 10))
ax.set_xlim(0, 14)
ax.set_ylim(0, 12)
ax.axis('off')

ax.text(7, 11.5, 'Complete Video Recommendation Pipeline',
        ha='center', fontsize=16, fontweight='bold')

# User request
ax.add_patch(mpatches.FancyBboxPatch((5, 10), 4, 1, boxstyle='round,pad=0.15',
             facecolor='#fff9c4', edgecolor='#f57f17', linewidth=2))
ax.text(7, 10.5, 'User Opens Homepage', ha='center', fontsize=11, fontweight='bold')

# Stage 1: Candidate Generation
ax.add_patch(mpatches.FancyBboxPatch((1, 7.5), 12, 2, boxstyle='round,pad=0.15',
             facecolor='#e3f2fd', edgecolor='#1565c0', linewidth=2))
ax.text(7, 9.1, 'STAGE 1: Candidate Generation', ha='center',
        fontsize=12, fontweight='bold', color='#1565c0')
ax.text(2, 8.4, 'CF Generator\n(Two-Tower)', ha='center', fontsize=9)
ax.text(5, 8.4, 'Trending\nGenerator', ha='center', fontsize=9)
ax.text(8, 8.4, 'Location\nGenerator', ha='center', fontsize=9)
ax.text(11, 8.4, 'Freshness\nGenerator', ha='center', fontsize=9)
ax.text(7, 7.7, '10B videos -> ~1,000-10,000 candidates', ha='center',
        fontsize=10, style='italic')

ax.annotate('', xy=(7, 7.5), xytext=(7, 10),
            arrowprops=dict(arrowstyle='->', lw=2, color='#1565c0'))

# Stage 2: Scoring/Ranking
ax.add_patch(mpatches.FancyBboxPatch((2, 4.5), 10, 2.5, boxstyle='round,pad=0.15',
             facecolor='#e8f5e9', edgecolor='#2e7d32', linewidth=2))
ax.text(7, 6.6, 'STAGE 2: Scoring / Ranking', ha='center',
        fontsize=12, fontweight='bold', color='#2e7d32')
ax.text(7, 5.9, 'Content-Based DNN Ranking Model', ha='center', fontsize=10)
ax.text(7, 5.3, 'User Features + Video Features -> P(relevant)', ha='center', fontsize=9)
ax.text(7, 4.8, '~1,000 candidates -> ~100-500 scored videos', ha='center',
        fontsize=10, style='italic')

ax.annotate('', xy=(7, 4.5), xytext=(7, 7.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#2e7d32'))

# Stage 3: Re-Ranking
ax.add_patch(mpatches.FancyBboxPatch((3, 1.5), 8, 2.5, boxstyle='round,pad=0.15',
             facecolor='#fce4ec', edgecolor='#c62828', linewidth=2))
ax.text(7, 3.6, 'STAGE 3: Re-Ranking', ha='center',
        fontsize=12, fontweight='bold', color='#c62828')
ax.text(7, 2.9, 'Diversity (MMR) + Business Rules', ha='center', fontsize=10)
ax.text(7, 2.3, 'Content policy | Regional | Freshness | Creator diversity',
        ha='center', fontsize=9)
ax.text(7, 1.7, '~100 videos -> 20-50 FINAL recommendations', ha='center',
        fontsize=10, style='italic')

ax.annotate('', xy=(7, 1.5), xytext=(7, 4.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#c62828'))

# Final output
ax.add_patch(mpatches.FancyBboxPatch((4.5, 0.2), 5, 0.8, boxstyle='round,pad=0.15',
             facecolor='#fff9c4', edgecolor='#f57f17', linewidth=2))
ax.text(7, 0.6, 'Homepage: 20-50 Recommended Videos', ha='center',
        fontsize=11, fontweight='bold')

ax.annotate('', xy=(7, 1.0), xytext=(7, 1.5),
            arrowprops=dict(arrowstyle='->', lw=2, color='#f57f17'))

plt.tight_layout()
plt.show()

## 6. Training the Ranker: Key Decisions

In [ ]:
# === Key Training Decisions for the Ranking Model ===

print("=" * 70)
print("KEY TRAINING DECISIONS FOR THE RANKING MODEL")
print("=" * 70)

decisions = [
    {
        "question": "What training data do we use?",
        "answer": "(user, video) pairs with label 1 (relevant) or 0 (not relevant)",
        "detail": "Relevant = user liked OR watched >= 50%. Negative = impressed but no engagement.",
        "pitfall": "Do NOT use random negatives for ranking (unlike candidate gen). Use actual impressions."
    },
    {
        "question": "How do we handle class imbalance?",
        "answer": "Focal loss, class weights, or negative down-sampling",
        "detail": "In practice, 90-99% of impressions are not relevant (negative).",
        "pitfall": "Pure accuracy is misleading. A model predicting all-negative gets 95% accuracy!"
    },
    {
        "question": "How do we avoid position bias?",
        "answer": "De-bias training data or model position as a feature",
        "detail": "Videos shown in position #1 get more clicks just because they're first.",
        "pitfall": "Without correction, the model learns 'position 1 is always clicked' -- not useful!"
    },
    {
        "question": "How often do we retrain?",
        "answer": "Daily for the full model, hourly for fine-tuning the last layers",
        "detail": "User preferences change. New content appears. Trends shift.",
        "pitfall": "Stale models quickly degrade. YouTube retrains models continuously."
    },
    {
        "question": "How do we evaluate before deployment?",
        "answer": "Offline: mAP, precision@k, diversity. Online: A/B test with CTR, watch time.",
        "detail": "Offline gains don't always translate to online gains (distribution shift).",
        "pitfall": "Never deploy based on offline metrics alone. Always A/B test."
    }
]

for d in decisions:
    print(f"\n  Q: {d['question']}")
    print(f"  A: {d['answer']}")
    print(f"  Detail: {d['detail']}")
    print(f"  Pitfall: {d['pitfall']}")

## Key Takeaways

1. **Ranking model** is a DNN that takes concatenated user + video features and predicts relevance. It can afford to be heavy because it only scores ~1,000 candidates.

2. **Feature interactions** are critical -- the power of DNNs is learning cross-feature patterns like "young mobile users prefer short gaming videos."

3. **Diversity** prevents echo chambers. MMR (Maximal Marginal Relevance) balances relevance and diversity by penalizing similarity to already-selected items.

4. **Re-ranking** applies business rules that ML cannot learn: content policy, regional restrictions, freshness boosts, creator diversity.

5. **Training decisions** matter: use actual impressions (not random negatives), handle class imbalance, correct for position bias, retrain frequently.

6. The three-stage pipeline (candidate generation -> ranking -> re-ranking) is a **speed vs. accuracy trade-off** that makes serving 10B videos in 200ms possible.

---

### Next: Notebook 4 -- Complete Mock Interview Walkthrough

In [ ]:
# ============================================================
# COACH — Session End  (do not remove this cell)
# ============================================================
from coach.notebook_widgets import render_session_end
render_session_end(_SESSION)